In [1]:
import numpy as np
import pandas as pd

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
df = pd.read_csv('../data/raw/ncr_ride_bookings.csv')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 21 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Date                               150000 non-null  object 
 1   Time                               150000 non-null  object 
 2   Booking ID                         150000 non-null  object 
 3   Booking Status                     150000 non-null  object 
 4   Customer ID                        150000 non-null  object 
 5   Vehicle Type                       150000 non-null  object 
 6   Pickup Location                    150000 non-null  object 
 7   Drop Location                      150000 non-null  object 
 8   Avg VTAT                           139500 non-null  float64
 9   Avg CTAT                           102000 non-null  float64
 10  Cancelled Rides by Customer        10500 non-null   float64
 11  Reason for cancelling by Customer  1050

In [5]:
rows, columns = df.shape
print(f"Rows:{rows}, Columns:{columns}")

Rows:150000, Columns:21


In [6]:
# to_datetime

df['Date'] = pd.to_datetime(df['Date'])

In [7]:
df.isna().sum()

Date                                      0
Time                                      0
Booking ID                                0
Booking Status                            0
Customer ID                               0
Vehicle Type                              0
Pickup Location                           0
Drop Location                             0
Avg VTAT                              10500
Avg CTAT                              48000
Cancelled Rides by Customer          139500
Reason for cancelling by Customer    139500
Cancelled Rides by Driver            123000
Driver Cancellation Reason           123000
Incomplete Rides                     141000
Incomplete Rides Reason              141000
Booking Value                         48000
Ride Distance                         48000
Driver Ratings                        57000
Customer Rating                       57000
Payment Method                        48000
dtype: int64

### Avg VTAT

In [9]:
df['Avg VTAT'].describe()

count    139500.000000
mean          8.456352
std           3.773564
min           2.000000
25%           5.300000
50%           8.300000
75%          11.300000
max          20.000000
Name: Avg VTAT, dtype: float64

In [10]:
# since there is no big outliers, filling the null values with the mean

df['Avg VTAT'].fillna(df['Avg VTAT'].mean(), inplace=True)

### Avg CTAT

In [12]:
df['Avg CTAT'].describe()

count    102000.000000
mean         29.149636
std           8.902577
min          10.000000
25%          21.600000
50%          28.800000
75%          36.800000
max          45.000000
Name: Avg CTAT, dtype: float64

In [13]:
df['Avg CTAT'].fillna(df['Avg CTAT'].mean(), inplace=True)

### Cancelled Rides by Customer and Driver

In [15]:
df['Cancelled Rides by Customer'].fillna(0, inplace=True)

In [16]:
df['Cancelled Rides by Driver'].fillna(0, inplace=True)

### Reasons

In [18]:
# checking if there are any null values for the reasons columns with the cancelled_ride true/1

In [19]:
df[df['Cancelled Rides by Customer'] == 1]['Reason for cancelling by Customer'].isnull().sum()

0

In [20]:
df[df['Cancelled Rides by Driver'] == 1]['Driver Cancellation Reason'].isnull().sum()

0

### Incomplete Rides and Reasons

In [22]:
# if the ride was cancelled already by driver/customer then marking it as incomplete ride

mask = (df['Cancelled Rides by Customer'] == 1) | (df['Cancelled Rides by Driver'] == 1) | (df['Booking Status'] == 'No Driver Found')
df.loc[mask, 'Incomplete Rides'] = 1

In [23]:
# converting the complete ride as False/0

df['Incomplete Rides'].fillna(0, inplace=True)

In [24]:
cust_mask = df['Cancelled Rides by Customer']==1

df.loc[cust_mask, 'Incomplete Rides Reason'] = 'Customer Cancellation'

In [25]:
driver_mask = df['Cancelled Rides by Driver']==1

df.loc[driver_mask, 'Incomplete Rides Reason'] = 'Driver Cancellation'

In [26]:
no_driver_mask = df['Booking Status'] == 'No Driver Found'

df.loc[no_driver_mask, 'Incomplete Rides Reason'] = 'No Driver Found'

In [27]:
# checking if the ride is complete and reason is null

df[df['Incomplete Rides'] == 1]['Incomplete Rides Reason'].isnull().sum()

0

### Ratings

In [29]:
df.head()

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,8.456352,29.149636,0.0,NaN,0.0,NaN,1.0,No Driver Found,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.900000,14.000000,0.0,NaN,0.0,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.400000,25.800000,0.0,NaN,0.0,NaN,0.0,NaN,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.100000,28.500000,0.0,NaN,0.0,NaN,0.0,NaN,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.300000,19.600000,0.0,NaN,0.0,NaN,0.0,NaN,737.0,48.21,4.1,4.3,UPI


In [30]:
# checking if there is null values in ratings even the ride has been completed

rating_mask = (df['Incomplete Rides']==0) & ((df['Driver Ratings'].isna()) | (df['Customer Rating'].isna()))

df.loc[rating_mask]

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method


In [31]:
df = df.rename(columns={
    'Date': 'date',
    'Time': 'time',
    'Booking ID': 'booking_id',
    'Booking Status': 'status',
    'Customer ID': 'customer_id',
    'Vehicle Type': 'vehicle_type',
    'Pickup Location': 'pickup_location',
    'Drop Location': 'drop_location',
    'Avg VTAT': 'avg_vtat',
    'Avg CTAT': 'avg_ctat',
    'Cancelled Rides by Customer': 'customer_cancel',
    'Reason for cancelling by Customer': 'customer_cancel_reason',
    'Cancelled Rides by Driver': 'driver_cancel',
    'Driver Cancellation Reason': 'driver_cancel_reason',
    'Incomplete Rides': 'incomplete_rides',
    'Incomplete Rides Reason': 'incomplete_rides_reason',
    'Booking Value': 'fare_price',
    'Ride Distance': 'ride_distance',
    'Driver Ratings': 'driver_rating',
    'Customer Rating': 'customer_rating',
    'Payment Method': 'pay_method'
})

In [32]:
df.to_csv('../data/processed/uber.csv', index=False)